In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
import sys
import pyspark.sql.functions as F
from src.config import data_paths, filter_map
from src.data_ingestion import DataLoader,DataMerger,Preprocessor
from src.logger import logger
from src.exception import M5Exception

In [4]:
data_paths

{'bronze': {'raw_paths': {'sales': 'C:\\m5_forecasting\\m5-demand-forecasting\\data\\bronze/raw/sales.csv',
   'calendar': 'C:\\m5_forecasting\\m5-demand-forecasting\\data\\bronze/raw/calendar.csv',
   'price': 'C:\\m5_forecasting\\m5-demand-forecasting\\data\\bronze/raw/sell_prices.csv'},
  'processed_data': {'sales_long': 'C:\\m5_forecasting\\m5-demand-forecasting\\data\\bronze/processed_data/sales_long',
   'joined_data': 'C:\\m5_forecasting\\m5-demand-forecasting\\data\\bronze/processed_data/joined_data'}},
 'silver': {}}

In [45]:
data = pd.read_parquet(data_paths['bronze']['processed_data']['joined_data'])
data = data.loc[data['state_id']=="CA"].sort_values(by=['id','yearweek'])
data.head()

,id,item_id,dept_id,cat_id,store_id,yearweek,weekly_sales,sell_price,snap_CA_days,snap_TX_days,...,f_event_christmas,f_event_memorialday,f_event_laborday,f_event_mothers_day,f_event_eidaladha,f_event_superbowl,f_event_presidentsday,f_event_easter,f_event_orthodoxchristmas,state_id
0,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,201104,3.0,2.0,0,0,...,0,0,0,0,0,0,0,0,0,CA
2,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,201105,9.0,2.0,6,4,...,0,0,0,0,0,1,0,0,0,CA
4,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,201106,7.0,2.0,4,5,...,0,0,0,0,0,0,0,0,0,CA
6,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,201107,10.0,2.0,0,1,...,0,0,0,0,0,0,0,0,0,CA
8,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,201108,14.0,2.0,0,0,...,0,0,0,0,0,0,1,0,0,CA


In [46]:
#how many products[items], customers[stores], Categories?
data['item_id'].nunique(), data['store_id'].nunique(),data['cat_id'].nunique()

(3049, 4, 3)

In [72]:
3049*4

12196

In [47]:
data.groupby(['cat_id'],as_index=False).agg(unique_ids=('id','nunique'),
                                          avg_sales = ('weekly_sales','mean'),
                                          total_sales = ('weekly_sales','sum'),
                                          sales_std = ('weekly_sales','std'))\
                                    .sort_values(by=['unique_ids','avg_sales'],ascending=False)\
                                    .assign(total_sales_m = lambda df:df['total_sales']/1e6,
                                            sales_pct = lambda df: (df['total_sales']/df['total_sales'].sum()) * 100,
                                            id_pct = lambda df: (df['unique_ids']/df['unique_ids'].sum())*100)

,cat_id,unique_ids,avg_sales,total_sales,sales_std,total_sales_m,sales_pct,id_pct
0,FOODS,5748,15.536083,19535863.0,37.176896,19.535863,66.911163,47.130207
2,HOUSEHOLD,4188,6.938276,6565267.0,14.197455,6.565267,22.486319,34.339128
1,HOBBIES,2260,6.096856,3095587.0,13.271080,3.095587,10.602517,18.530666


- 66% of sales, 47% of series are from Foods
- 22% of sales, 34% of series are from HouseHold
- 10% of sales, 18% of series are from Hobbies

In [44]:
data.shape

(2711425, 42)

In [62]:
#leading zero check
cumsum_sales = data.groupby("id")["weekly_sales"].cumsum()
initial_zero_sales= (data.loc[cumsum_sales.eq(0),"id"].drop_duplicates().reset_index(drop=True))
initial_zero_sales

Series([], Name: id, dtype: object)

In [79]:
#missing week check
week_date = pd.to_datetime(
    data["yearweek"] + "1",
    format="%G%V%u"
)

week_diff = week_date.groupby(data["id"]).diff().dt.days

# missing_week_ids = (
#     data.loc[week_diff.gt(7), "id"]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )

missing_gaps = data.loc[
    week_diff.gt(7),
    ["id", "yearweek"]
]
missing_gaps

,id,yearweek
556,FOODS_1_001_CA_1,201653
1357247,FOODS_1_001_CA_2,201653
1114,FOODS_1_001_CA_3,201653
2008469,FOODS_1_001_CA_4,201653
1672,FOODS_1_002_CA_1,201653
...,...,...
1045136,HOUSEHOLD_2_515_CA_4,201653
1045694,HOUSEHOLD_2_516_CA_1,201653
1046250,HOUSEHOLD_2_516_CA_2,201653
2360854,HOUSEHOLD_2_516_CA_3,201653


- for all series we don't have 201653.

In [82]:
# missing week fill

data["weekdate"] = pd.to_datetime(
    data["yearweek"] + "1",
    format="%G%V%u"
)

In [1]:
data.shape

NameError: name 'data' is not defined

In [85]:
temp = (
    data.set_index("weekdate")
        .groupby("id")
        .resample("W-MON")
        .asfreq()
        .drop(columns="id")
        .reset_index()
)

MemoryError: Unable to allocate 310. MiB for an array with shape (30, 2711425) and data type int32